# ETL: Tipo de cambio USD→MXN (Extract → Transform → Load a MariaDB)

Versión para **Google Colab** del ejercicio en `recursos/etl-tipo-cambio/` del repo del curso.

- **Extract:** API pública de [Frankfurter](https://api.frankfurter.app) (tipos de cambio históricos, sin API key).
- **Transform:** limpieza, columnas derivadas y validación de calidad de datos con `pandas`.
- **Load:** conexión a la instancia de MariaDB en GCP (ver `environment/mariadb-vscode-setup.md`) con **upsert idempotente**.

### Sobre la conexión a MariaDB desde Colab

Colab no tiene una IP fija (cambia cada vez que se reinicia el entorno), así que la
"Opción A" del setup original (restringir el firewall a tu IP) **no funciona bien aquí**.
En este notebook se usa en su lugar un **túnel IAP** (`gcloud compute start-iap-tunnel`):
no requiere abrir el puerto 3306 a internet ni administrar llaves SSH, solo un login de
Google y permisos de IAM sobre el proyecto. Es la forma recomendada de conectarse a una
VM desde un entorno con IP dinámica como Colab.

## 0. Configuración

Editar estos valores antes de correr el notebook.

In [ ]:
# --- Datos del ejercicio ---
MONEDA_BASE = "USD"
MONEDAS_DESTINO = ["MXN"]
DIAS_HISTORIA = 90

# --- Datos de tu instancia de GCP (ver environment/mariadb-vscode-setup.md) ---
GCP_PROJECT_ID = "tu-proyecto-gcp"
GCP_ZONE = "us-central1-a"
VM_NAME = "mariadb-curso"

# --- Credenciales de MariaDB (usuario compartido de la clase, ver recursos/mariadb/) ---
DB_USER = "big_data_user"
DB_PASSWORD = "Example123"
DB_NAME = "curso_bigdata"

# Puerto local en Colab al que se reenviará el 3306 remoto vía el túnel IAP
LOCAL_TUNNEL_PORT = 3307


## 1. Instalar dependencias

In [ ]:
!pip install -q requests pandas mysql-connector-python

## 2. EXTRACT — Traer los datos de la API pública

Misma lógica que `extract.py` del repo, adaptada a celdas de notebook.

In [ ]:
import requests
import sys
from datetime import date, timedelta

BASE_URL = "https://api.frankfurter.app"

def extraer_serie_historica(moneda_base, monedas_destino, fecha_inicio, fecha_fin):
    simbolos = ",".join(monedas_destino)
    url = f"{BASE_URL}/{fecha_inicio.isoformat()}..{fecha_fin.isoformat()}"
    params = {"from": moneda_base, "to": simbolos}

    try:
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"ERROR al llamar a la API: {e}", file=sys.stderr)
        print("Si el problema persiste, revisar si el endpoint cambió a "
              "https://api.frankfurter.dev/v1/", file=sys.stderr)
        raise

    datos = resp.json()
    if "rates" not in datos or not datos["rates"]:
        raise ValueError(f"La API respondió sin datos para {fecha_inicio}..{fecha_fin}: {datos}")
    return datos


fecha_fin = date.today()
fecha_inicio = fecha_fin - timedelta(days=DIAS_HISTORIA)

datos_crudos = extraer_serie_historica(MONEDA_BASE, MONEDAS_DESTINO, fecha_inicio, fecha_fin)
print(f"OK: {len(datos_crudos['rates'])} fechas extraídas de {fecha_inicio} a {fecha_fin}")

In [ ]:
# Vista rápida del JSON crudo (capa "bronze", tal como lo entrega la API)
list(datos_crudos["rates"].items())[:3]

## 3. TRANSFORM — Aplanar, calcular columnas derivadas y validar calidad

Misma lógica que `transform.py` del repo.

In [ ]:
import pandas as pd

def a_tabla_larga(datos):
    moneda_base = datos["base"]
    filas = []
    for fecha_str, tasas_del_dia in datos["rates"].items():
        for moneda_destino, tasa in tasas_del_dia.items():
            filas.append({
                "fecha": fecha_str,
                "moneda_origen": moneda_base,
                "moneda_destino": moneda_destino,
                "tasa": tasa,
            })
    df = pd.DataFrame(filas)
    df["fecha"] = pd.to_datetime(df["fecha"])
    return df.sort_values(["moneda_destino", "fecha"]).reset_index(drop=True)


def calcular_columnas_derivadas(df):
    df = df.copy()
    grupo = df.groupby("moneda_destino")["tasa"]
    df["variacion_diaria"] = grupo.diff()
    df["variacion_pct"] = grupo.pct_change() * 100
    df["promedio_movil_7d"] = grupo.transform(lambda s: s.rolling(window=7, min_periods=1).mean())
    return df


df = a_tabla_larga(datos_crudos)
df = calcular_columnas_derivadas(df)
df.tail(10)

In [ ]:
def validar_calidad(df):
    errores = []

    if df["tasa"].isnull().any():
        errores.append("Hay valores nulos en la columna 'tasa'.")
    if (df["tasa"] <= 0).any():
        errores.append("Hay tasas de cambio menores o iguales a cero.")

    duplicados = df.duplicated(subset=["fecha", "moneda_origen", "moneda_destino"]).sum()
    if duplicados > 0:
        errores.append(f"Hay {duplicados} filas duplicadas por (fecha, origen, destino).")

    variacion_extrema = df["variacion_pct"].abs() > 15
    if variacion_extrema.any():
        errores.append(f"Hay {variacion_extrema.sum()} filas con variación diaria > 15%.")

    if errores:
        for e in errores:
            print(f"  - {e}")
        raise ValueError("Los datos no pasaron la validación de calidad. Revisar antes de cargar.")
    print("Validación de calidad: OK")


validar_calidad(df)

### Vista rápida: tipo de cambio en el periodo (aprovechando que estamos en un notebook)

In [ ]:
import matplotlib.pyplot as plt

for moneda in df["moneda_destino"].unique():
    sub = df[df["moneda_destino"] == moneda]
    plt.figure(figsize=(10, 4))
    plt.plot(sub["fecha"], sub["tasa"], label=f"{MONEDA_BASE}/{moneda} diario")
    plt.plot(sub["fecha"], sub["promedio_movil_7d"], label="Promedio móvil 7d", linestyle="--")
    plt.title(f"Tipo de cambio {MONEDA_BASE}/{moneda}")
    plt.xlabel("Fecha")
    plt.ylabel("Tasa")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 4. Conectar a MariaDB desde Colab (túnel IAP)

**Prerequisito:** una regla de firewall que permita tráfico del rango de IAP hacia el
puerto 3306. Si ya creaste la instancia con `recursos/mariadb/crear_firewall_y_instancia.sh`
(el flujo recomendado del repo), **esta regla ya existe** — sáltate el bloque de abajo y
continúa directo a la siguiente celda.

Si tu instancia se creó de otra forma y no tienes la regla todavía:

```bash
gcloud compute firewall-rules create allow-iap-mariadb \
  --network=default \
  --direction=INGRESS \
  --action=ALLOW \
  --rules=tcp:3306 \
  --source-ranges=35.235.240.0/20 \
  --target-tags=mariadb-server
```

El rango `35.235.240.0/20` es fijo (pertenece a Google, no a tu Colab), así que esta regla
no hay que tocarla otra vez aunque Colab te dé una IP distinta cada sesión — esa es
justamente la diferencia con la Opción A de `mariadb-vscode-setup.md`.

In [ ]:
# Autenticarse con gcloud dentro de Colab (abre un flujo de login de Google)
!gcloud auth login --no-launch-browser

In [ ]:
!gcloud config set project {GCP_PROJECT_ID}

In [ ]:
# Abrir el túnel IAP en segundo plano: reenvía localhost:LOCAL_TUNNEL_PORT -> VM:3306
import subprocess
import time

comando_tunel = [
    "gcloud", "compute", "start-iap-tunnel", VM_NAME, "3306",
    f"--local-host-port=localhost:{LOCAL_TUNNEL_PORT}",
    f"--zone={GCP_ZONE}",
]
proceso_tunel = subprocess.Popen(comando_tunel, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

time.sleep(6)  # dar tiempo a que el túnel termine de establecerse
print("Túnel IAP iniciado (revisa que no haya errores abajo si la conexión falla más adelante).")

## 5. Crear la tabla destino (si no existe)

In [ ]:
import mysql.connector

SQL_CREATE_TABLA = """
CREATE TABLE IF NOT EXISTS tipo_cambio (
    fecha               DATE            NOT NULL,
    moneda_origen       VARCHAR(3)      NOT NULL,
    moneda_destino      VARCHAR(3)      NOT NULL,
    tasa                DECIMAL(14,6)   NOT NULL,
    variacion_diaria    DECIMAL(14,6)   NULL,
    variacion_pct       DECIMAL(8,4)    NULL,
    promedio_movil_7d   DECIMAL(14,6)   NULL,
    cargado_en          TIMESTAMP       DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    PRIMARY KEY (fecha, moneda_origen, moneda_destino)
) ENGINE=InnoDB;
"""

conexion = mysql.connector.connect(
    host="127.0.0.1",
    port=LOCAL_TUNNEL_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
)
cursor = conexion.cursor()
cursor.execute(SQL_CREATE_TABLA)
conexion.commit()
print("Tabla lista.")

## 6. LOAD — Cargar con upsert idempotente

Misma lógica que `load.py` del repo.

In [ ]:
SQL_UPSERT = """
    INSERT INTO tipo_cambio
        (fecha, moneda_origen, moneda_destino, tasa, variacion_diaria, variacion_pct, promedio_movil_7d)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        tasa = VALUES(tasa),
        variacion_diaria = VALUES(variacion_diaria),
        variacion_pct = VALUES(variacion_pct),
        promedio_movil_7d = VALUES(promedio_movil_7d)
"""

df_cargar = df.where(pd.notnull(df), None)

filas = [
    (
        row.fecha.date(),
        row.moneda_origen,
        row.moneda_destino,
        float(row.tasa),
        float(row.variacion_diaria) if row.variacion_diaria is not None else None,
        float(row.variacion_pct) if row.variacion_pct is not None else None,
        float(row.promedio_movil_7d) if row.promedio_movil_7d is not None else None,
    )
    for row in df_cargar.itertuples(index=False)
]

cursor.executemany(SQL_UPSERT, filas)
conexion.commit()
print(f"OK: {cursor.rowcount} filas afectadas (inserts + updates).")

## 7. Verificar la carga

In [ ]:
verificacion = pd.read_sql(
    "SELECT * FROM tipo_cambio ORDER BY fecha DESC LIMIT 10", conexion
)
verificacion

## 8. Limpieza

Cerrar la conexión y el túnel, y apagar la VM si ya no se va a usar (para no gastar
crédito de GCP innecesariamente).

In [ ]:
cursor.close()
conexion.close()
proceso_tunel.terminate()
print("Conexión y túnel cerrados.")

In [ ]:
# Opcional: apagar la VM desde el propio notebook
# !gcloud compute instances stop {VM_NAME} --zone={GCP_ZONE}